# Hypothesis H2: Tier-1 Vendors Have Lower Defect Rates Than Tier-2 and Tier-3 Vendors

This notebook evaluates whether vendor tier is associated with supplier quality performance.

## Business objective
Compare vendor defect rates across Tier-1, Tier-2, and Tier-3 suppliers and determine whether Tier-1 vendors perform significantly better.

## Hypotheses
- **H0:** There is no statistically significant difference in defect rates among Tier-1, Tier-2, and Tier-3 vendors.
- **H1:** Tier-1 vendors exhibit significantly lower defect rates than Tier-2 and Tier-3 vendors.

## Statistical approach
- One-Way ANOVA
- ANOVA assumption checks
- Tukey HSD post-hoc comparisons if the ANOVA is significant

## Dataset used
- Source requested: `vendors_cleaned.csv`
- Source available in this project: `data/cleaned/vendors_clean.csv`

**Modeling note:** the vendor file is a monthly snapshot table. To preserve group independence for ANOVA, this notebook first aggregates monthly defect rates to a single average defect rate per vendor.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'vendors_clean.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Reports' / 'h2_vendor_quality'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUTPUT_DIR

In [ ]:
vendors = pd.read_csv(DATA_PATH)
vendors.head()

## Prepare the analysis dataset
We keep the tier and defect fields needed for the test, standardize tier labels, and aggregate the monthly snapshots to one record per vendor.

In [ ]:
analysis_df = vendors[['vendor_id', 'vendor_name', 'vendor_tier', 'snapshot_month', 'defect_rate_pct']].copy()
analysis_df['snapshot_month'] = pd.to_datetime(analysis_df['snapshot_month'], errors='coerce')
analysis_df['vendor_tier'] = analysis_df['vendor_tier'].astype(str).str.strip().str.title()
analysis_df['defect_rate_pct'] = pd.to_numeric(analysis_df['defect_rate_pct'], errors='coerce')
analysis_df = analysis_df[analysis_df['vendor_tier'].isin(['Tier-1', 'Tier-2', 'Tier-3'])]
analysis_df = analysis_df.dropna(subset=['defect_rate_pct'])

vendor_level = (
    analysis_df
    .groupby(['vendor_id', 'vendor_name', 'vendor_tier'], as_index=False)
    .agg(
        avg_defect_rate_pct=('defect_rate_pct', 'mean'),
        median_defect_rate_pct=('defect_rate_pct', 'median'),
        monthly_observations=('defect_rate_pct', 'size')
    )
)

vendor_level.head()

In [ ]:
vendor_counts = vendor_level['vendor_tier'].value_counts().sort_index().rename('vendor_count')
vendor_counts.to_frame()

## Mean comparison table

In [ ]:
mean_comparison = (
    vendor_level
    .groupby('vendor_tier')['avg_defect_rate_pct']
    .agg(
        vendor_count='size',
        mean_defect_rate='mean',
        median_defect_rate='median',
        std_defect_rate='std',
        min_defect_rate='min',
        max_defect_rate='max'
    )
    .assign(
        ci95_half_width=lambda d: 1.96 * d['std_defect_rate'] / np.sqrt(d['vendor_count'])
    )
    .sort_index()
)

mean_comparison

In [ ]:
mean_comparison.to_csv(OUTPUT_DIR / 'mean_comparison_table.csv')
print(f'Saved mean comparison table to: {OUTPUT_DIR / "mean_comparison_table.csv"}')

## Validate ANOVA assumptions
We check:
1. Independent groups: handled by using one aggregated record per vendor.
2. Approximate normality within each tier: Shapiro-Wilk test.
3. Homogeneity of variances: Levene's test.

Because business datasets can be large and imperfect, these tests are used as diagnostics rather than as strict pass/fail gates.

In [ ]:
tier_order = ['Tier-1', 'Tier-2', 'Tier-3']
groups = {
    tier: vendor_level.loc[vendor_level['vendor_tier'] == tier, 'avg_defect_rate_pct']
    for tier in tier_order
}

shapiro_results = []
for tier, values in groups.items():
    sample_for_test = values.sample(5000, random_state=42) if len(values) > 5000 else values
    stat, p_value = stats.shapiro(sample_for_test)
    shapiro_results.append({
        'vendor_tier': tier,
        'n_used_for_test': len(sample_for_test),
        'shapiro_statistic': stat,
        'shapiro_p_value': p_value
    })

shapiro_table = pd.DataFrame(shapiro_results)
levene_stat, levene_p = stats.levene(*[groups[tier] for tier in tier_order], center='median')

shapiro_table

In [ ]:
assumption_summary = pd.DataFrame([
    {
        'test': 'Levene variance equality test',
        'statistic': levene_stat,
        'p_value': levene_p,
        'interpretation': 'Variances differ across tiers' if levene_p < 0.05 else 'No strong evidence of unequal variances'
    }
])

assumption_summary

## Perform One-Way ANOVA

In [ ]:
anova_stat, anova_p = stats.f_oneway(*[groups[tier] for tier in tier_order])

grand_mean = vendor_level['avg_defect_rate_pct'].mean()
ss_between = sum(len(values) * (values.mean() - grand_mean) ** 2 for values in groups.values())
ss_total = ((vendor_level['avg_defect_rate_pct'] - grand_mean) ** 2).sum()
eta_squared = ss_between / ss_total

anova_results = pd.DataFrame([
    {
        'test': 'One-Way ANOVA',
        'f_statistic': anova_stat,
        'p_value': anova_p,
        'eta_squared': eta_squared,
        'significant_at_0_05': anova_p < 0.05
    }
])

anova_results

In [ ]:
anova_results.to_csv(OUTPUT_DIR / 'anova_results.csv', index=False)
print(f'Saved ANOVA results to: {OUTPUT_DIR / "anova_results.csv"}')

## Post-hoc comparisons: Tukey HSD
Run pairwise comparisons to identify which tiers differ from each other.

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=vendor_level['avg_defect_rate_pct'],
    groups=vendor_level['vendor_tier'],
    alpha=0.05
)

# Convert the Tukey summary into a DataFrame.
tukey_table = pd.DataFrame(
    tukey.summary().data[1:],
    columns=tukey.summary().data[0]
)

for col in ['meandiff', 'p-adj', 'lower', 'upper']:
    tukey_table[col] = pd.to_numeric(tukey_table[col], errors='coerce')

tukey_table['reject'] = tukey_table['reject'].astype(str)
tukey_table

In [ ]:
tukey_table.to_csv(OUTPUT_DIR / 'tukey_hsd_results.csv', index=False)
print(f'Saved Tukey HSD results to: {OUTPUT_DIR / "tukey_hsd_results.csv"}')

## Box plots by vendor tier

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(
    data=vendor_level,
    x='vendor_tier',
    y='avg_defect_rate_pct',
    hue='vendor_tier',
    order=tier_order,
    palette=['#2E7D32', '#F9A825', '#C62828'],
    dodge=False,
    legend=False,
    ax=ax
)
sns.stripplot(
    data=vendor_level,
    x='vendor_tier',
    y='avg_defect_rate_pct',
    order=tier_order,
    color='black',
    alpha=0.35,
    size=3,
    ax=ax
)
ax.set_title('Average Vendor Defect Rates by Vendor Tier')
ax.set_xlabel('Vendor Tier')
ax.set_ylabel('Average Defect Rate')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'boxplot_vendor_tier_defect_rate.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved box plot to: {plot_path}')

## Vendor quality report
This section converts the statistical output into a procurement-facing summary and recommendation.

In [ ]:
tier_means = mean_comparison['mean_defect_rate'].to_dict()

if anova_p < 0.05 and tier_means['Tier-1'] < tier_means['Tier-2'] and tier_means['Tier-1'] < tier_means['Tier-3']:
    decision = 'Reject H0 in favor of H1'
    headline = 'Tier-1 vendors have the lowest average defect rates and the differences are statistically significant.'
else:
    decision = 'Fail to reject H0'
    headline = 'The analysis does not provide enough evidence that Tier-1 vendors outperform the lower tiers.'

report_lines = [
    '# Vendor Quality Report: Hypothesis H2',
    '',
    '## Executive summary',
    headline,
    '',
    '## Key statistics',
    f"- Tier-1 mean defect rate: {tier_means['Tier-1']:.4f}",
    f"- Tier-2 mean defect rate: {tier_means['Tier-2']:.4f}",
    f"- Tier-3 mean defect rate: {tier_means['Tier-3']:.4f}",
    f'- One-Way ANOVA F-statistic: {anova_stat:.4f}',
    f'- One-Way ANOVA p-value: {anova_p:.4e}',
    f'- Effect size (eta-squared): {eta_squared:.4f}',
    f'- Decision: {decision}',
    '',
    '## Assumption check summary',
    '- Independence was improved by aggregating monthly records to one record per vendor.',
    f"- Shapiro-Wilk p-values by tier: {', '.join(f'{row.vendor_tier}={row.shapiro_p_value:.4f}' for row in shapiro_table.itertuples())}",
    f'- Levene test p-value: {levene_p:.4e}',
    '- Variance equality is not supported, so the ANOVA result should be interpreted with that caveat.',
    '',
    '## Procurement recommendations',
    '- Favor Tier-1 vendors for quality-sensitive categories and warranty-sensitive products.',
    '- Launch corrective action plans for Tier-2 and Tier-3 vendors with the highest average defect rates.',
    '- Add defect-rate thresholds and review checkpoints to supplier scorecards and renewal decisions.',
    '- Use this result together with spend, lead time, and concentration risk before changing sourcing allocations.'
]

report_text = '
'.join(report_lines)
report_path = OUTPUT_DIR / 'vendor_quality_report.md'
report_path.write_text(report_text, encoding='utf-8')

print(report_text)
print(f'
Saved report to: {report_path}')

## Conclusion
If you rerun this notebook, it will regenerate the analysis outputs in `Reports/h2_vendor_quality/` for easy sharing with procurement stakeholders.